In [1]:
import itertools, csv, time
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt

param range
epochs = [100, 200, 300]
batch_size = [8, 16]
learning_rate = [0.001, 0.01]

aug param range
hsv_h = 0
hsv_s = [0, 0.35, 0.7]
hsv_v = [0.2, 0.4]
mosaic = [0.5, 1.0]
degrees = 360

download the model
model = YOLO("yolov8n-seg.pt")

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET_YAML = "./17_03_dataset/data.yaml"
RESULTS_CSV  = "./grid_search_results.csv"
RUN_DIR      = "./grid_search_runs"

# ── Parameter grid ────────────────────────────────────────────────────────────
# All combinations of these values will be trained.
# Everything not listed here stays at YOLO default.
PARAM_GRID = {
    "epochs":  [50],
    "batch":   [8, 16],
    "lr0":     [0.001, 0.01],
    "hsv_h":   [0],              # fixed — no hue shift for thermal images
    "hsv_s":   [0, 0.35, 0.7],
    "hsv_v":   [0.2, 0.4],
    "mosaic":  [0.5, 1.0],
    "degrees": [360],            # fixed — full rotation invariance
}

# ── Baseline: pure YOLO defaults, no overrides ────────────────────────────────
BASELINE = {
    "epochs": 100, "batch": 16, "lr0": 0.01,
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "mosaic": 1.0, "degrees": 0.0,
}

total = 1
for v in PARAM_GRID.values():
    total *= len(v)
print(f"Grid combinations : {total}")
print(f"Total runs        : {total + 1}  (grid + 1 baseline)")

Grid combinations : 48
Total runs        : 49  (grid + 1 baseline)


In [ ]:
# ── CSV helpers ───────────────────────────────────────────────────────────────
CSV_FIELDS = [
    "run_name", "run_type",
    "epochs", "batch", "lr0",
    "hsv_h", "hsv_s", "hsv_v", "mosaic", "degrees",
    "seg_map50", "seg_map50_95", "seg_precision", "seg_recall",
    "box_map50", "box_map50_95", "box_precision", "box_recall",
    "train_time_s", "inference_ms_per_img",
    "status",
]

def init_csv():
    if not Path(RESULTS_CSV).exists():
        with open(RESULTS_CSV, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

def already_done(run_name):
    if not Path(RESULTS_CSV).exists():
        return False
    with open(RESULTS_CSV) as f:
        return any(row["run_name"] == run_name for row in csv.DictReader(f))

def save_row(row):
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

def get_metrics(m):
    """Extract seg + box metrics from a val() result object."""
    def safe(fn):
        try: return round(float(fn()), 4)
        except: return None
    return {
        "seg_map50":            safe(lambda: m.seg.map50),
        "seg_map50_95":         safe(lambda: m.seg.map),
        "seg_precision":        safe(lambda: m.seg.p[0]),
        "seg_recall":           safe(lambda: m.seg.r[0]),
        "box_map50":            safe(lambda: m.box.map50),
        "box_map50_95":         safe(lambda: m.box.map),
        "box_precision":        safe(lambda: m.box.p[0]),
        "box_recall":           safe(lambda: m.box.r[0]),
        "inference_ms_per_img": safe(lambda: m.speed["inference"]),
    }

In [ ]:
# ── Training function ─────────────────────────────────────────────────────────
def run_training(run_name, run_type, params):
    if already_done(run_name):
        print(f"  SKIP (already in CSV): {run_name}")
        return

    print(f"\n{'='*60}\n  RUN  : {run_name}\n  PARAMS: {params}")

    row = {"run_name": run_name, "run_type": run_type, **params,
           "seg_map50": None, "seg_map50_95": None,
           "seg_precision": None, "seg_recall": None,
           "box_map50": None, "box_map50_95": None,
           "box_precision": None, "box_recall": None,
           "train_time_s": None, "inference_ms_per_img": None,
           "status": "error"}
    try:
        model = YOLO("yolov8n-seg.pt")

        t0 = time.time()
        model.train(
            data=DATASET_YAML, project=RUN_DIR,
            name=run_name, exist_ok=True, **params
        )
        row["train_time_s"] = round(time.time() - t0, 1)

        best = YOLO(f"{RUN_DIR}/{run_name}/weights/best.pt")
        row.update(get_metrics(best.val(data=DATASET_YAML, split="test")))
        row["status"] = "ok"

    except Exception as e:
        row["status"] = f"error: {e}"
        print(f"  ERROR: {e}")

    save_row(row)
    print(f"  seg_mAP50={row['seg_map50']}  seg_mAP50-95={row['seg_map50_95']}  status={row['status']}")

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
init_csv()

# 1. Baseline first
run_training("baseline", "baseline", BASELINE)

# 2. All grid combinations
keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*PARAM_GRID.values()))

for i, combo in enumerate(combos, 1):
    params = dict(zip(keys, combo))
    name = (
        f"ep{params['epochs']}"
        f"_bs{params['batch']}"
        f"_lr{str(params['lr0']).replace('.','')}"
        f"_s{str(params['hsv_s']).replace('.','')}"
        f"_v{str(params['hsv_v']).replace('.','')}"
        f"_mos{str(params['mosaic']).replace('.','')}"
    )
    print(f"[{i:03d}/{len(combos)}]", end="")
    run_training(name, "grid", params)

print("\nGrid search complete.")

## Results

In [ ]:
df = pd.read_csv(RESULTS_CSV)
df_ok = df[df["status"] == "ok"].copy()

print(f"Total runs: {len(df)}  |  Successful: {len(df_ok)}  |  Failed: {len(df) - len(df_ok)}")
df_ok.sort_values("seg_map50_95", ascending=False).reset_index(drop=True)

In [ ]:
# ── Bar chart: effect of each parameter on seg mAP50-95 ──────────────────────
params_to_plot = ["epochs", "batch", "lr0", "hsv_s", "hsv_v", "mosaic"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

baseline_val = df[df["run_type"] == "baseline"]["seg_map50_95"].values
baseline_val = baseline_val[0] if len(baseline_val) else None

grid_df = df_ok[df_ok["run_type"] == "grid"]

for ax, param in zip(axes, params_to_plot):
    grouped = grid_df.groupby(param)["seg_map50_95"]
    means = grouped.mean().sort_index()
    stds  = grouped.std().sort_index()

    ax.bar(means.index.astype(str), means.values, yerr=stds.values,
           capsize=4, color="steelblue", edgecolor="white", linewidth=0.5)

    if baseline_val is not None:
        ax.axhline(baseline_val, color="tomato", linewidth=1.2,
                   linestyle="--", label=f"baseline ({baseline_val:.4f})")
        ax.legend(fontsize=7)

    ax.set_title(param, fontweight="bold")
    ax.set_ylabel("seg mAP50-95")

fig.suptitle("Effect of each parameter on seg mAP50-95 (mean ± std)\nRed dashed = baseline",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("grid_search_results.png", dpi=150, bbox_inches="tight")
plt.show()